# Raw to Silver - Biblioteca Dataset

Lee CSVs de la capa raw (ADLS Gen2), explora los datos, detecta y corrige problemas de calidad,
y escribe la capa silver en formato Delta.

In [ ]:
STORAGE_ACCOUNT_NAME = "stadatademo2026"
RAW_CONTAINER = "raw"
SILVER_CONTAINER = "silver"

RAW_BASE_PATH = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net"
SILVER_BASE_PATH = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net"

RAW_FILE_NAMES = ["autores", "libros", "usuarios", "prestamos"]

In [ ]:
STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope="storage", key="account-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_KEY
)

## 1. Lectura de CSVs desde la capa Raw

In [ ]:
raw_dataframes = {}

for file_name in RAW_FILE_NAMES:
    raw_path = f"{RAW_BASE_PATH}/{file_name}.csv"
    raw_dataframes[file_name] = spark.read.option("header", True).option("inferSchema", True).csv(raw_path)

raw_authors = raw_dataframes["autores"]
raw_books = raw_dataframes["libros"]
raw_users = raw_dataframes["usuarios"]
raw_loans = raw_dataframes["prestamos"]

## 2. Exploración de datos

In [ ]:
for table_name, dataframe in raw_dataframes.items():
    print(f"\n{'='*60}")
    print(f"Tabla: {table_name}")
    print(f"Registros: {dataframe.count()} | Columnas: {len(dataframe.columns)}")
    print(f"{'='*60}")
    dataframe.printSchema()
    dataframe.show(5, truncate=False)

In [ ]:
from pyspark.sql.functions import col, count, when, isnan, isnull

for table_name, dataframe in raw_dataframes.items():
    print(f"\nNulos por columna en '{table_name}':")
    null_counts = dataframe.select(
        [count(when(isnull(col(column_name)), column_name)).alias(column_name) for column_name in dataframe.columns]
    )
    null_counts.show(truncate=False)

## 3. Detección de problemas de calidad

In [ ]:
from pyspark.sql.functions import lower, trim, collect_set

quality_log_entries = []

### 3.1 Géneros con casing inconsistente en libros

In [ ]:
distinct_genres = raw_books.select("genero").distinct().orderBy("genero")
print("Valores únicos de 'genero' antes de normalizar:")
distinct_genres.show(50, truncate=False)

genre_count_before = distinct_genres.count()

normalized_genre_count = raw_books.select(lower(trim(col("genero"))).alias("genero_normalizado")).distinct().count()

inconsistent_genre_count = genre_count_before - normalized_genre_count
has_genre_inconsistencies = inconsistent_genre_count > 0

if has_genre_inconsistencies:
    message = f"Detectados {inconsistent_genre_count} géneros duplicados por casing ({genre_count_before} únicos -> {normalized_genre_count} tras normalizar)"
    print(message)
    quality_log_entries.append({"tabla": "libros", "problema": "casing_inconsistente_genero", "detalle": message, "accion": "Normalizado a minúsculas con trim"})

### 3.2 Duplicados en préstamos

In [ ]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

LOAN_DEDUP_COLUMNS = ["id_libro", "id_usuario", "fecha_prestamo"]

total_loans_before = raw_loans.count()

dedup_window = Window.partitionBy(LOAN_DEDUP_COLUMNS).orderBy("id")
loans_with_row_number = raw_loans.withColumn("row_number", row_number().over(dedup_window))

duplicate_loan_count = loans_with_row_number.filter(col("row_number") > 1).count()
has_duplicate_loans = duplicate_loan_count > 0

if has_duplicate_loans:
    duplicate_percentage = round(duplicate_loan_count / total_loans_before * 100, 1)
    message = f"Detectados {duplicate_loan_count} préstamos duplicados ({duplicate_percentage}%) por combinación libro+usuario+fecha"
    print(message)
    quality_log_entries.append({"tabla": "prestamos", "problema": "registros_duplicados", "detalle": message, "accion": "Eliminados duplicados conservando primer registro por id"})

print(f"\nEjemplos de duplicados:")
loans_with_row_number.filter(col("row_number") > 1).show(10, truncate=False)

### 3.3 Fecha de devolución nula en préstamos devueltos

In [ ]:
is_returned = col("estado") == "devuelto"
has_null_return_date = isnull(col("fecha_devolucion"))

returned_without_date = raw_loans.filter(is_returned & has_null_return_date)
returned_without_date_count = returned_without_date.count()

has_missing_return_dates = returned_without_date_count > 0

if has_missing_return_dates:
    total_returned = raw_loans.filter(is_returned).count()
    missing_percentage = round(returned_without_date_count / total_returned * 100, 1)
    message = f"Detectados {returned_without_date_count} préstamos con estado 'devuelto' sin fecha_devolucion ({missing_percentage}% de devueltos)"
    print(message)
    quality_log_entries.append({"tabla": "prestamos", "problema": "fecha_devolucion_nula_en_devueltos", "detalle": message, "accion": "Se asigna fecha_devolucion = fecha_prestamo + 14 días como estimación"})

print(f"\nEjemplos:")
returned_without_date.show(10, truncate=False)

## 4. Limpieza y transformación

### 4.1 Autores y usuarios (sin problemas detectados, solo tipado)

In [ ]:
from pyspark.sql.functions import to_date

clean_authors = raw_authors.select(
    col("id").cast("int").alias("id"),
    trim(col("nombre")).alias("nombre"),
    trim(col("nacionalidad")).alias("nacionalidad"),
    col("anio_nacimiento").cast("int").alias("anio_nacimiento")
)

clean_users = raw_users.select(
    col("id").cast("int").alias("id"),
    trim(col("nombre")).alias("nombre"),
    trim(col("email")).alias("email"),
    to_date(col("fecha_registro")).alias("fecha_registro"),
    trim(col("ciudad")).alias("ciudad")
)

### 4.2 Libros - normalizar género

In [ ]:
from pyspark.sql.functions import initcap

clean_books = raw_books.select(
    col("id").cast("int").alias("id"),
    trim(col("titulo")).alias("titulo"),
    col("id_autor").cast("int").alias("id_autor"),
    initcap(trim(col("genero"))).alias("genero"),
    col("anio_publicacion").cast("int").alias("anio_publicacion"),
    col("paginas").cast("int").alias("paginas")
)

### 4.3 Préstamos - eliminar duplicados y corregir fechas

In [ ]:
from pyspark.sql.functions import date_add, coalesce, lit

ESTIMATED_LOAN_DURATION_DAYS = 14

deduplicated_loans = loans_with_row_number.filter(col("row_number") == 1).drop("row_number")

is_returned_with_null_date = (col("estado") == "devuelto") & isnull(col("fecha_devolucion"))
estimated_return_date = date_add(to_date(col("fecha_prestamo")), ESTIMATED_LOAN_DURATION_DAYS)

clean_loans = deduplicated_loans.select(
    col("id").cast("int").alias("id"),
    col("id_libro").cast("int").alias("id_libro"),
    col("id_usuario").cast("int").alias("id_usuario"),
    to_date(col("fecha_prestamo")).alias("fecha_prestamo"),
    when(is_returned_with_null_date, estimated_return_date).otherwise(to_date(col("fecha_devolucion"))).alias("fecha_devolucion"),
    trim(col("estado")).alias("estado")
)

## 5. Verificación post-limpieza

In [ ]:
clean_dataframes = {
    "autores": clean_authors,
    "libros": clean_books,
    "usuarios": clean_users,
    "prestamos": clean_loans
}

print("Resumen post-limpieza:")
print(f"{'Tabla':<15} {'Registros raw':>15} {'Registros clean':>17}")
print("-" * 50)
for table_name in RAW_FILE_NAMES:
    raw_count = raw_dataframes[table_name].count()
    clean_count = clean_dataframes[table_name].count()
    print(f"{table_name:<15} {raw_count:>15} {clean_count:>17}")

In [ ]:
print("\nGéneros únicos en libros (post-limpieza):")
clean_books.select("genero").distinct().orderBy("genero").show(50, truncate=False)

remaining_returned_without_date = clean_loans.filter((col("estado") == "devuelto") & isnull(col("fecha_devolucion"))).count()
print(f"Préstamos devueltos sin fecha_devolucion tras limpieza: {remaining_returned_without_date}")

## 6. Escritura en capa Silver (formato Delta)

In [ ]:
for table_name, clean_dataframe in clean_dataframes.items():
    silver_path = f"{SILVER_BASE_PATH}/{table_name}"
    clean_dataframe.write.format("delta").mode("overwrite").save(silver_path)
    print(f"Escrito: {silver_path} ({clean_dataframe.count()} registros)")

## 7. Log de calidad

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime

QUALITY_LOG_SCHEMA = StructType([
    StructField("tabla", StringType(), False),
    StructField("problema", StringType(), False),
    StructField("detalle", StringType(), False),
    StructField("accion", StringType(), False),
    StructField("fecha_ejecucion", StringType(), False)
])

execution_timestamp = datetime.now().isoformat()

quality_log_rows = [
    (entry["tabla"], entry["problema"], entry["detalle"], entry["accion"], execution_timestamp)
    for entry in quality_log_entries
]

quality_log_dataframe = spark.createDataFrame(quality_log_rows, schema=QUALITY_LOG_SCHEMA)

quality_log_path = f"{SILVER_BASE_PATH}/_quality_log"
quality_log_dataframe.write.format("delta").mode("append").save(quality_log_path)

print(f"\nLog de calidad guardado en: {quality_log_path}")
print(f"Problemas registrados: {len(quality_log_entries)}\n")
quality_log_dataframe.show(truncate=False)